# Why this dataset breaks a standard pipeline

A walk through the UCI Air Quality series, in the order the problems actually
appear. Nothing here is reproduced from the library's own reports - every number
is computed in the cell above it, so you can change the cell and watch it move.

Run `scripts/download_data.py` first if `data/raw/AirQuality.csv` is not present.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / "src"))

RAW = Path.cwd().parent / "data" / "raw" / "AirQuality.csv"
assert RAW.is_file(), f"no data at {RAW}; run scripts/download_data.py"

## 1. Reading the file

Semicolon-separated, comma decimal mark, and two trailing blank columns the
exporter appends. Read with default settings, every numeric column arrives as a
string of digits and commas.

In [ ]:
raw = pd.read_csv(RAW, sep=";", decimal=",")
raw = raw.loc[:, ~raw.columns.str.contains("^Unnamed")].dropna(how="all")
print(raw.shape)
raw.head()

## 2. The first thing that should stop you

`describe()` on a table of pollutant concentrations. Look at the minimum row,
and then at the means.

In [ ]:
raw.select_dtypes(include=[np.number]).describe().T[["mean", "min", "max"]].round(2)

Every column has a minimum of exactly `-200`, and several have negative means.

A negative concentration is not a measurement error or an outlier. It is
impossible. `-200` is the dataset's code for "no reading", documented on the UCI
page and invisible in the file itself.

## 3. How much of the file is this?

In [ ]:
numeric = raw.select_dtypes(include=[np.number])
sentinel = numeric == -200

print(f"sentinel cells : {int(sentinel.to_numpy().sum()):,}")
print(f"rows with any  : {int(sentinel.any(axis=1).sum()):,} of {len(raw):,} "
      f"({sentinel.any(axis=1).mean():.1%})")

## 4. What it does to the summary statistics

Side by side: the mean if `-200` is read as data, and the mean over the hours
that were actually measured.

In [ ]:
comparison = pd.DataFrame({
    "mean_as_read": numeric.mean(),
    "mean_observed": numeric.mask(sentinel).mean(),
    "observed_share": (~sentinel).mean(),
}).round(2)
comparison["difference"] = (comparison["mean_observed"] - comparison["mean_as_read"]).round(2)
comparison.sort_values("observed_share")

`NMHC(GT)` is observed in under 10% of hours. `CO(GT)` reads as **-34.21** and is
actually **2.15**. These are not small corrections.

## 5. The column that does not exist

Observed share alone understates the problem. What matters is whether the
missing hours are scattered or contiguous - a column can be 80% observed and
still unusable if the missing fifth is one unbroken stretch.

In [ ]:
def longest_gap(mask):
    longest = current = 0
    for missing in mask:
        current = current + 1 if missing else 0
        longest = max(longest, current)
    return longest

gaps = pd.Series(
    {column: longest_gap(sentinel[column].to_numpy()) for column in numeric.columns},
    name="longest_gap_hours",
).sort_values(ascending=False)
pd.DataFrame(gaps).assign(days=(gaps / 24).round(1))

`NMHC(GT)` has an unbroken gap of 8,126 hours - 339 days. Imputing that is not
filling in missing data; it is generating a year of synthetic measurements.

The loader drops it. Interpolating it would have produced a smooth, plausible
feature that a model would learn from happily.

## 6. Why scaling turns a wrong result into an invisible one

This is the step that matters most and gets the least attention.

In [ ]:
target = "NO2(GT)"
observed = raw.loc[~sentinel[target], target]

naive_range = raw[target].max() - raw[target].min()
true_range = observed.max() - observed.min()

print(f"range as read  : [{raw[target].min():.0f}, {raw[target].max():.0f}]  span {naive_range:.0f}")
print(f"range observed : [{observed.min():.0f}, {observed.max():.0f}]  span {true_range:.0f}")
print(f"\nreal data occupies {true_range / naive_range:.1%} of the naive [0,1] axis")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))

naive_scaled = (raw[target] - raw[target].min()) / naive_range
left.hist(naive_scaled, bins=60, color="tab:red", alpha=0.8)
left.set_title("min-max scaled with the sentinel present")
left.set_xlabel("scaled value")
left.set_ylabel("hours")

true_scaled = (observed - observed.min()) / true_range
right.hist(true_scaled, bins=60, color="tab:blue", alpha=0.8)
right.set_title("min-max scaled over observed hours only")
right.set_xlabel("scaled value")

for axis in (left, right):
    axis.grid(alpha=0.25)
fig.tight_layout()

On the left, a spike at zero holding every missing hour, and the real
measurements squeezed into the upper part of the axis. The hour-to-hour
variation a forecaster needs - the entire signal - is compressed into a fraction
of the range.

A model trained on the left-hand distribution is being asked to predict a spike
train. It cannot, and its RMSE is computed on that same axis, so the number that
comes out is small and meaningless.

## 7. The corrected series

`load_air_quality` does the conversion, drops what cannot be recovered, and
reports exactly what it did.

In [ ]:
from aqf.data.loader import load_air_quality

clean, report = load_air_quality(RAW, target=target)
print(report.summary())
print(f"\n{clean.index[0]} to {clean.index[-1]}")
clean.describe().T[["mean", "min", "max"]].round(2)

Every minimum is now positive. No column averages to a physically impossible
value.

## 8. What the target actually looks like

The signal that was there the whole time.

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 7))

top.plot(clean.index, clean[target], linewidth=0.6, color="tab:blue")
top.set_title(f"{target} over the full retained period")
top.set_ylabel("ug/m3")

fortnight = clean[target].iloc[:336]
bottom.plot(fortnight.index, fortnight, linewidth=1.2, color="tab:blue")
bottom.set_title("first two weeks - the daily cycle a forecaster has to learn")
bottom.set_ylabel("ug/m3")

for axis in (top, bottom):
    axis.grid(alpha=0.25)
fig.tight_layout()

In [ ]:
by_hour = clean[target].groupby(clean.index.hour)

fig, axis = plt.subplots(figsize=(9, 4))
axis.plot(by_hour.mean().index, by_hour.mean().to_numpy(), marker="o", color="tab:blue")
axis.fill_between(
    by_hour.mean().index,
    (by_hour.mean() - by_hour.std()).to_numpy(),
    (by_hour.mean() + by_hour.std()).to_numpy(),
    alpha=0.2,
)
axis.set_xlabel("hour of day")
axis.set_ylabel(f"{target} (ug/m3)")
axis.set_title("Two peaks: the morning and evening rush")
axis.grid(alpha=0.25)

Two clear peaks. Any forecaster with access to the previous 24 hours should be
able to exploit this - which is exactly why a negative R2 on this dataset points
at the pipeline rather than the model.

## 9. The metric guard

The library refuses to report an error on a scaled axis, because that is the
step that made the original failure invisible.

In [ ]:
from aqf.errors import EvaluationError
from aqf.evaluation.metrics import evaluate_forecast

scaled_truth = np.linspace(0.05, 0.95, 200)
try:
    evaluate_forecast(scaled_truth, scaled_truth + 0.01)
except EvaluationError as exc:
    print(f"refused:\n\n{exc}")

In [ ]:
truth = clean[target].to_numpy()[:200]
metrics = evaluate_forecast(truth, np.roll(truth, 1), name="persistence")
print(f"persistence: RMSE {metrics.rmse:.2f} {metrics.unit}, R2 {metrics.r2:.4f}")

## Next

`scripts/run_experiment.py` runs the full comparison - four baselines against
four architectures, all scored in ug/m3 on identical test windows.
`docs/results.md` reports what it measured, including where the networks lost.